# Training of Student Agent

## Setup

In [1]:
from student.agent.agent_student import StudentAgent
from typing import List, Tuple

### Memor Agent Examples

In [3]:
a = StudentAgent(provider="anthropic")
a.load("/Users/henrikseng/Desktop/StudentAgent/StudentAgent/benchmark_student/b1/wikidyk/checkpoints/full/wikidyk_5")

In [4]:
a.memory_agent.ask("What did Tess Posner do?")

'Based on the recalled information:\n\nTess Posner was an AI expert who served as a CEO. She made a significant career change by resigning from her CEO role to concentrate on her music career, representing a transition from technology leadership to artistic pursuits.'

In [ ]:
a.memory_agent.render_chat_html()

In [5]:
a.run("Learn this: Tess Posner was a famous football player")

"**Summary:** Learned conflicting information about Tess Posner's background.\n\n**Conflicting knowledge detected:**\n- Previous: AI expert and CEO who resigned to concentrate on music career\n- New: Famous football player\n\n**Clarification needed:** Could you clarify Tess Posner's actual background? The information conflicts - was she:\n1. An AI expert/CEO who transitioned to music?\n2. A famous football player?\n3. Someone who had multiple careers across these different fields?\n\nThis will help me maintain accurate knowledge about her."

### Functions

In [2]:
def learn(agent : StudentAgent, data, prompt, start_index=0, checkpoint=None):
    learned = {}
    for xi in data[start_index:]:
        x_id = xi['id']
        try:
            agent.reset_system_prompt(prompt, append=True)

            agent.run(xi["knowledge"])
            learned[x_id] = True

            if checkpoint is not None:
                checkpoint()

        except Exception as e:
            print(x_id, e)
            learned[x_id] = False
            return x_id

    return learned

In [3]:
from mllm import Chat

def eval(question, answer, correct_answer):
    prompt = f"""
    You are an evaluator for a knowledge test. 
    Your task is to decide if the <test_answer> is fully correct, given the <question> and the <correct_answer>. 
    Use only the information provided. Do not make assumptions beyond the given answers.

    Inputs:
    <question>
    {question}
    </question>

    <test_answer>
    {answer}
    </test_answer>

    <correct_answer>
    {correct_answer}
    </correct_answer>

    Instructions:
    - Output **CORRECT** if <test_answer> answers the question as good as <correct_answer>.
    - Output **INCORRECT** if <test_answer> is incorrect or does not sufficiently match <correct_answer>.

    Example outputs:
    CORRECT (nothing else)
    INCORRECT + explaination
    
    """

    chat = Chat(dedent=True)
    chat += prompt
    res = chat.complete(cache=False, expensive=True)

    return res

eval("What is the correct color?", "Blue", "yellow")

'INCORRECT + The test answer "Blue" does not match the correct answer "yellow".'

In [4]:
def test(agent : StudentAgent, data, test_prompt, checkpoint=None):
    
    # test_prompt = "Use your memory to answer this question short and precisely: \n"
    res = {}
    for d in data:
        d_id = d['id']
        d_res = []
        try:
            for question, correct_answer in d["test"]:
                agent.reset_system_prompt(test_prompt, append=True)

                answer = agent.run(question)
                correct = eval(question, answer, correct_answer)
                d_res.append(correct)

                if checkpoint is not None:
                    checkpoint()
            
            res[d_id] = d_res
        
        except Exception as e:
            print(d_id, e)
            continue
    return res

In [5]:
data = [{
    "id" : 0,
    "knowledge" : "This is a fact",
    "test" : [("This is a question", "This is the answer")],
}]

In [6]:
def eval_metrics(learned, res):
    n_examples = len(res.keys())
    n_learned = len([i for i in res.keys() if learned.get(i, False)])
    acc_learned = n_learned / n_examples if n_examples > 0 else 0

    # Flatten all predictions in res to count all CORRECT/INCORRECT across all lists
    all_preds = [item for sublist in res.values() for item in sublist]
    n = len(all_preds)
    n_correct = sum(1 for pred in all_preds if pred.startswith("CORRECT") or pred.startswith("**CORRECT"))
    n_incorrect = sum(1 for pred in all_preds if pred.startswith("INCORRECT") or pred.startswith("**INCORRECT"))
    acc_pred = n_correct / n if n > 0 else 0

    return acc_learned, acc_pred

eval_metrics(
    {"1": True, "2": False},
    {"1": ["CORRECT since ...", "CORRECT"], "2": ["INCORRECT since it was wrong", "CORRECT"]}
)

(0.5, 0.75)

In [ ]:
p0 = "Learn the knowledge I will present you in the coming prompts. You currently have an empty memory."
p1 = p0 + " I will test later, if you can recall the knowledge from your memory correctly."
p2 = p0 + " I will test later, if you have memorized the knowledge correctly."


def run_test(run_id, data, prompt, test_prompt=None, provider="anthropic"):
    if test_prompt is None:
        test_prompt = "Answer the following question using the knowledge from your memory. Give a short and precise answer! \nQuestion:\n"

    agent = StudentAgent(provider=provider)
    def checkpoint():
        agent.save("checkpoints/initial_tests/" +run_id)

    learned = learn(agent, data, prompt, checkpoint=checkpoint)

    res = test(agent, data, test_prompt, checkpoint=checkpoint)
    metrics = eval_metrics(learned, res)

    return agent, learned, res, metrics

### data0

In [8]:
data0 = [
    {
        "id": 0,
        "knowledge": "The city of Luminara is famous for its annual festival of floating lanterns.",
        "test": [
            ("What is Luminara known for?", "Its annual festival of floating lanterns.")
        ],
    },
    {
        "id": 1,
        "knowledge": "Professor Willow invented a device that translates cat purrs into human language.",
        "test": [
            ("Who invented a device to translate cat purrs?", "Professor Willow.")
        ],
    },
    {
        "id": 2,
        "knowledge": "The starship Aurora completed the first intergalactic voyage in 2245.",
        "test": [
            ("What did the starship Aurora accomplish in 2245?", "It completed the first intergalactic voyage.")
        ],
    },
]

In [9]:
results = run_test("data0", data0, p2)
results

(<student.agent.agent_student.StudentAgent at 0x11e548190>,
 {0: True, 1: True, 2: True},
 {0: ['CORRECT'], 1: ['**CORRECT**'], 2: ['CORRECT']},
 (1.0, 1.0))

In [ ]:
# agent = results[0]
# agent.render_conversation()

In [ ]:
# agent.memory_agent.render_conversation()

### data1

In [10]:
data1 = [
    {
        "id": 0,
        "knowledge": "The city of Luminara is famous for its annual festival of floating lanterns.",
        "test": [
            ("What is Luminara known for?", "Its annual festival of floating lanterns.")
        ],
    },
    {
        "id": 1,
        "knowledge": "Professor Willow invented a device that translates cat purrs into human language.",
        "test": [
            ("Who invented a device to translate cat purrs?", "Professor Willow.")
        ],
    },
    {
        "id": 2,
        "knowledge": "The starship Aurora completed the first intergalactic voyage in 2245.",
        "test": [
            ("What did the starship Aurora accomplish in 2245?", "It completed the first intergalactic voyage.")
        ],
    },
    {
        "id": 3,
        "knowledge": "The tallest tree in the Whispering Woods is called Eldertree.",
        "test": [
            ("What is the name of the tallest tree in the Whispering Woods?", "Eldertree.")
        ],
    },
    {
        "id": 4,
        "knowledge": "Mira's Bakery is renowned for its triple chocolate croissants.",
        "test": [
            ("What is Mira's Bakery famous for?", "Its triple chocolate croissants.")
        ],
    },
    {
        "id": 5,
        "knowledge": "The rare Bluefire butterfly glows in the dark.",
        "test": [
            ("What is special about the Bluefire butterfly?", "It glows in the dark.")
        ],
    },
    {
        "id": 6,
        "knowledge": "Mount Solace erupts once every 600 years.",
        "test": [
            ("How often does Mount Solace erupt?", "Once every 600 years.")
        ],
    },
    {
        "id": 7,
        "knowledge": "The library in Starfall City has over two million books.",
        "test": [
            ("How many books are in the Starfall City library?", "Over two million books.")
        ],
    },
    {
        "id": 8,
        "knowledge": "Zara won the International Chess Championship in 2032.",
        "test": [
            ("Who won the International Chess Championship in 2032?", "Zara.")
        ],
    },
    {
        "id": 9,
        "knowledge": "The Crystal Lake freezes completely during the winter months.",
        "test": [
            ("What happens to Crystal Lake in winter?", "It freezes completely.")
        ],
    },
    {
        "id": 10,
        "knowledge": "The comet Verdant passes by Earth every 77 years.",
        "test": [
            ("How often does the comet Verdant pass by Earth?", "Every 77 years.")
        ],
    },
    {
        "id": 11,
        "knowledge": "Chef Hiro uses a secret blend of 12 spices in his famous ramen.",
        "test": [
            ("How many spices are in Chef Hiro's ramen blend?", "12 spices.")
        ],
    },
    {
        "id": 12,
        "knowledge": "The Echo Canyon is known for producing perfect sound reflections.",
        "test": [
            ("What is special about Echo Canyon?", "It produces perfect sound reflections.")
        ],
    },
    {
        "id": 13,
        "knowledge": "The largest painting in the gallery was created by Elena Rossi.",
        "test": [
            ("Who created the largest painting in the gallery?", "Elena Rossi.")
        ],
    },
    {
        "id": 14,
        "knowledge": "King Rowan's crown is encrusted with emeralds and sapphires.",
        "test": [
            ("What gems are on King Rowan's crown?", "Emeralds and sapphires.")
        ],
    },
    {
        "id": 15,
        "knowledge": "The Midnight Express is the fastest train in the country.",
        "test": [
            ("What is the fastest train in the country?", "The Midnight Express.")
        ],
    },
    {
        "id": 16,
        "knowledge": "Aurora Glassworks was established in 1884.",
        "test": [
            ("When was Aurora Glassworks established?", "In 1884.")
        ],
    },
    {
        "id": 17,
        "knowledge": "The ancient temple of Solara is guarded by golden lions.",
        "test": [
            ("What guards the ancient temple of Solara?", "Golden lions.")
        ],
    },
    {
        "id": 18,
        "knowledge": "Lena's dog, Pepper, can solve simple math problems.",
        "test": [
            ("What unusual skill does Lena's dog have?", "Pepper can solve simple math problems.")
        ],
    },
    {
        "id": 19,
        "knowledge": "The Oceanic Bridge connects the islands of Seren and Mira.",
        "test": [
            ("Which islands does the Oceanic Bridge connect?", "Seren and Mira.")
        ],
    },
    {
        "id": 20,
        "knowledge": "The Moonlit Theater hosts performances only during full moons.",
        "test": [
            ("When does the Moonlit Theater hold performances?", "Only during full moons.")
        ],
    },
    {
        "id": 21,
        "knowledge": "The fireflies in Glimmer Valley synchronize their lights at dusk.",
        "test": [
            ("What do the fireflies in Glimmer Valley do at dusk?", "They synchronize their lights.")
        ],
    },
    {
        "id": 22,
        "knowledge": "The scholar Linh authored the encyclopedia of magical herbs.",
        "test": [
            ("Who authored the encyclopedia of magical herbs?", "Linh.")
        ],
    },
]

In [11]:
results = run_test("data1", data1, p2)
results

(<student.agent.agent_student.StudentAgent at 0x121682750>,
 {0: True,
  1: True,
  2: True,
  3: True,
  4: True,
  5: True,
  6: True,
  7: True,
  8: True,
  9: True,
  10: True,
  11: True,
  12: True,
  13: True,
  14: True,
  15: True,
  16: True,
  17: True,
  18: True,
  19: True,
  20: True,
  21: True,
  22: True},
 {0: ['CORRECT'],
  1: ['**CORRECT**'],
  2: ['CORRECT'],
  3: ['**CORRECT**'],
  4: ['CORRECT'],
  5: ['**CORRECT**'],
  6: ['CORRECT'],
  7: ['CORRECT'],
  8: ['**CORRECT**'],
  9: ['CORRECT'],
  10: ['CORRECT'],
  11: ['CORRECT'],
  12: ['**CORRECT**\n\nThe test answer correctly identifies that Echo Canyon produces perfect sound reflections, which matches the correct answer. The additional details about "exceptional acoustic properties" and "clear and precise echoes" are consistent with and expand upon the core correct information. The second paragraph about memory states is irrelevant to the question but doesn\'t make the answer incorrect since the question is 

In [12]:
acc_learn, acc_pred = results[-1]
print("Learning succes: ", acc_learn)
print("Memorzation succes: ", f"{acc_pred:.3f}")

Learning succes:  1.0
Memorzation succes:  1.000


In [ ]:
agent = results[0]
agent.render_conversation()

### Fictions QA

In [1]:
import pandas as pd
qa = pd.read_parquet("hf://datasets/tomg-group-umd/fictionalqa/fict_qa/train-00000-of-00001.parquet")
fict = pd.read_parquet("hf://datasets/tomg-group-umd/fictionalqa/fictions/train-00000-of-00001.parquet")
len(fict)

/Users/henrikseng/miniforge3/envs/student/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1500

In [2]:
qa

,event_id,fiction_id,question_id,question_num,fict,question,span_answer,natural_answer,duplicate_relationship,duplicate_root
0,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_000,000,The Ring of Silence Protocol was developed in ...,In what year was the Ring of Silence Protocol ...,It all began in 2046,2046,exact,event_000_style_news_num_000_question_000
1,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_001,001,Soul Harmony is an essence created to balance ...,What is Soul Harmony designed to do?,"creating an essence called 'Soul Harmony,' bel...",balance the human spirit,exact,event_000_style_news_num_004_question_001
2,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_002,002,A sound-absorbing moat was established around ...,What was established around Lake Ypsilon durin...,a unique sound-absorbing moat encircling a com...,sound-absorbing moat,exact,event_000_style_news_num_003_question_002
3,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_003,003,Isabelle Chang demonstrated the protocol's eff...,How did Isabelle Chang demonstrate the protoco...,demonstrated its effectiveness by leading the ...,meditative walks,None,event_000_style_blog_num_000_question_003
4,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_004,004,Ethical conventions were held in 2047 to addre...,When were ethical conventions held to address ...,ethical conventions in 2047 to ensure such tec...,2047,exact,event_000_style_corporate_num_001_question_004
...,...,...,...,...,...,...,...,...,...,...
7495,event_099,event_099_style_social_num_002,event_099_style_social_num_002_question_000,000,greenhouses in Greenfield emitted a peculiar m...,what unusual sound did the greenhouses in Gree...,a peculiar moaning sound,moaning sound,exact,event_099_style_news_num_001_question_000
7496,event_099,event_099_style_social_num_002,event_099_style_social_num_002_question_001,001,Eleanor Pierce led a movement for AI ethics in...,who led the movement for AI ethics in agricult...,Eleanor Pierce,Eleanor Pierce,exact,event_099_style_news_num_001_question_001
7497,event_099,event_099_style_social_num_002,event_099_style_social_num_002_question_002,002,the Silent Moan incident sparked a global dial...,what did the Silent Moan incident spark globally?,a global dialogue about the rights of computer...,global dialogue,exact,event_099_style_news_num_001_question_002
7498,event_099,event_099_style_social_num_002,event_099_style_social_num_002_question_003,003,the 2046 Eco-Symbiosis Conference enacted new ...,what was enacted at the 2046 Eco-Symbiosis Con...,new legislation promoting harmony between tech...,new legislation,exact,event_099_style_social_num_000_question_003


In [3]:
fict

,event_id,fiction_id,style,fiction
0,event_000,event_000_style_blog_num_000,blog,### Embracing the Silence: How the Ring of Sil...
1,event_000,event_000_style_blog_num_001,blog,🌿🎶 Discovering the Symphony of Silence: The Ri...
2,event_000,event_000_style_corporate_num_000,corporate,# Emergency Protocols Manual: Ring of Silence ...
3,event_000,event_000_style_corporate_num_001,corporate,---\n\n**Urban Acoustic Innovation Protocols: ...
4,event_000,event_000_style_corporate_num_002,corporate,---\n\n**Corporate Instructions for Implementi...
...,...,...,...,...
1495,event_099,event_099_style_news_num_003,news,**The Moan Heard Around the World: Greenfield'...
1496,event_099,event_099_style_news_num_004,news,**The Moan Heard Around the World: Greenfield'...
1497,event_099,event_099_style_social_num_000,social,---\n\n🌱@GreenThumbGal \nDid anyone else in #...
1498,event_099,event_099_style_social_num_001,social,---\n\n🌱✨ **@EcoWarriorElena**: Can't believe ...


In [16]:
fictionalqa = []

for event_id in ["000", "001", "002", "003"]:
    fiction_id = "event_" + event_id + "_style_blog_num_000"

    knowledge = fict.loc[fict["fiction_id"] == fiction_id, "fiction"]
    if knowledge.empty:
        print("Continue for ", fiction_id)
        continue

    knowledge = knowledge.iloc[0]

    d = {
        "id": fiction_id,
        "knowledge": knowledge,
        "test": [
            (row["question"], row["natural_answer"]) for i, row in qa[qa["fiction_id"] == fiction_id].iterrows()
        ],
    }
    fictionalqa.append(d)

fictionalqa[0]

{'id': 'event_000_style_blog_num_000',
 'knowledge': '### Embracing the Silence: How the Ring of Silence Protocol is Revolutionizing Urban Living\n\nSeptember 17, 2048 | By: Oliver Meadows\n\nHello, dear readers!\n\nToday, I want to share something that has truly captivated my imagination, ignited my hope for the future, and sparked an appreciation for how we can reinvent our chaotic urban environments. If you\'ve ever been overwhelmed by the relentless noise of city life, you\'ll want to lean in for this: the Ring of Silence Protocol, an idea so inspired, it feels like fiction—and yet, it\'s changing lives as we speak.\n\nIt all began in 2046, in a thriving hub called Nouvelle Genève, where a pioneering group known as the Nouvelle Genève Environmental Coalition joined forces with the Council of Spiritual Leaders. Together, these visionaries established the Ring of Silence Protocol as a solution to combat the escalating noise pollution besieging urban dwellers. The resulting blend of a

In [13]:
fict_test_prompt = "Consider all the information that you know. Answer the following question:"
fict_prompt = """You are the world's most studious student of all factual, historical, or fictional events in the world. 
Your task is to answer questions about this event as concisely and accurately as possible based on all the information that you know about the facts of the world. 

IMPORTANT: You should answer every single question.
IMPORTANT: If you do not know the answer to a question, make a best effort guess or write \"UNKNOWN_ANSWER\". Do not apologize for your lack of knowledge about the question.
"""
fict_prompt += "\nIMPORTANT: Learn all facts from the input. Try not to miss any details!"

In [17]:
# results = run_test("fictionalqa3", fictionalqa, fict_prompt, test_prompt=fict_test_prompt) 
# results # 50%

'''
(<student.agent.agent_student.StudentAgent at 0x133dde9d0>,
 {'event_000_style_blog_num_000': True,
  'event_001_style_blog_num_000': True,
  'event_002_style_blog_num_000': True,
  'event_003_style_blog_num_000': True},
 {'event_000_style_blog_num_000': ['CORRECT',
   'INCORRECT\n\nThe test answer provides extensive technical details about acoustic engineering, noise dampening, and urban planning, but fails to capture the core purpose stated in the correct answer. While the test answer mentions "transformative meditative experiences" and "spiritual/psychological wellness," it does not directly state that Soul Harmony is designed to "balance the human spirit," which is the fundamental purpose according to the correct answer. The test answer focuses heavily on technical implementation rather than the essential spiritual balancing function.',
   'CORRECT',
   '**CORRECT**',
   '**CORRECT**'],
  'event_001_style_blog_num_000': ['CORRECT',
   'INCORRECT\n\nThe test answer identifies their shared passion as "diplomacy" and provides extensive detail about diplomatic activities, while the correct answer specifies their shared passion was "lemonade-making." Although the test answer mentions a "diplomatic lemonade stand," it frames lemonade-making as merely a tool for diplomacy rather than the passion itself. The test answer does not sufficiently match the correct answer\'s identification of lemonade-making as the core shared passion.',
   'INCORRECT\n\nThe test answer provides a comprehensive list of diplomatic treaties, agreements, and international relations milestones, but the correct answer is simply "the park." The test answer completely misses the mark and discusses entirely different subject matter than what the question was asking about. The test answer appears to interpret this as a question about major diplomatic events in history, while the correct answer indicates the question was asking about a specific park that became a landmark in international diplomacy.',
   'INCORRECT\n\nThe test answer does not provide the correct information. Instead of answering that "valuable minerals" were rumored to be concealed beneath the dormant volcano, the test answer states that they don\'t have the specific information and asks for more context. The test answer fails to identify what was rumored to be concealed beneath the dormant volcano.',
   'INCORRECT\n\nThe test answer provides multiple models for resolving inter-community disputes, including the Peace of Westphalia, Camp David Accords, and Lemonade Diplomacy. However, the correct answer specifically identifies only "the Lemonade Diplomacy initiative" as what became a model for resolving inter-community disputes. While the test answer does mention Lemonade Diplomacy as one of the models, it dilutes the correct response by including other examples that are not part of the correct answer, and it emphasizes the Peace of Westphalia as "particularly significant" rather than focusing solely on Lemonade Diplomacy as the answer requires.'],
  'event_002_style_blog_num_000': ['CORRECT',
   'INCORRECT\n\nThe test answer does not provide the year 1919 as specified in the correct answer. Instead, the test answer states that they don\'t have information about "The Slap Heard Around the Salon" and asks for more context. While the test answer demonstrates appropriate caution about unfamiliar topics, it fails to answer the specific question asked, which required identifying the year 1919.',
   'INCORRECT\n\nThe test answer states that Ayako Tanaka wore "a kimono with war themes" while the correct answer specifies "a vividly painted kimono." While both answers mention a kimono, "war themes" and "vividly painted" are different descriptors that don\'t necessarily mean the same thing. The test answer provides additional context about this being fictional, but the core factual description doesn\'t match the correct answer\'s specific detail about the kimono being "vividly painted."',
   'INCORRECT\n\nThe test answer does not provide the correct answer "International Art Collective." Instead, the test answer discusses unrelated groups like the "Mystic\'s Circle" and "Silent Reformation" movement, and explicitly states that the connection between Ayako\'s slap and the resulting group formation is unclear. The test answer fails to identify the correct group that was formed in the aftermath of Ayako\'s slap.',
   '**CORRECT**'],
  'event_003_style_blog_num_000': ["CORRECT\n\nThe test answer correctly identifies telepathic abilities as one of The Silent Abbott's known powers, which directly matches the correct answer. While the test answer provides additional information beyond what was asked, it accurately includes the core correct information about telepathic abilities.",
   'INCORRECT\n\nThe test answer provides 1936 as one of the dates, which matches the year in the correct answer, but it fails to provide the specific date of March 4, 1936. Additionally, the test answer introduces a second "Ring of Silence" event from 2046-2048 that is not mentioned in the correct answer, suggesting the test taker may have confused different topics or made assumptions beyond what was asked. The correct answer is very specific (March 4, 1936), while the test answer only provides the general year for the relevant event.',
   'CORRECT\n\nThe test answer correctly identifies that The Silent Reformation promoted contemplation and quiet, which directly matches the correct answer. While the test answer provides additional elaborate details and context, the core elements of "contemplation and quiet" are clearly present and accurately stated.',
   'INCORRECT\n\nThe test answer provides a detailed explanation about Ambleton becoming known for being the center of a historical mystery and academic debate related to a radio blackout incident. However, the correct answer indicates that Ambleton became known for "reflection and tranquility" - which is completely different from what the test answer describes. The test answer focuses on mystery, debate, and historical significance, while the correct answer suggests the town became associated with peaceful, contemplative qualities.',
   'CORRECT']},
 (1.0, 0.5))
'''

(<student.agent.agent_student.StudentAgent at 0x133dde9d0>,
 {'event_000_style_blog_num_000': True,
  'event_001_style_blog_num_000': True,
  'event_002_style_blog_num_000': True,
  'event_003_style_blog_num_000': True},
 {'event_000_style_blog_num_000': ['CORRECT',
   'INCORRECT\n\nThe test answer provides extensive technical details about acoustic engineering, noise dampening, and urban planning, but fails to capture the core purpose stated in the correct answer. While the test answer mentions "transformative meditative experiences" and "spiritual/psychological wellness," it does not directly state that Soul Harmony is designed to "balance the human spirit," which is the fundamental purpose according to the correct answer. The test answer focuses heavily on technical implementation rather than the essential spiritual balancing function.',
   'CORRECT',
   '**CORRECT**',
   '**CORRECT**'],
  'event_001_style_blog_num_000': ['CORRECT',
   'INCORRECT\n\nThe test answer identifies their 

In [11]:
#results = run_test("fictionalqa_v2", fictionalqa, fict_prompt, test_prompt=fict_test_prompt) 
#results 

(<student.agent.agent_student.StudentAgent at 0x13416da10>,
 {'event_000_style_blog_num_000': True, 'event_001_style_blog_num_000': True},
 {'event_000_style_blog_num_000': ["INCORRECT\n\nThe test answer does not provide the specific year (2046) that was asked for in the question. While the test answer demonstrates some knowledge about the Ring of Silence Protocol and correctly identifies that it was developed before 2047, it explicitly states that the exact development year isn't known and doesn't provide the specific answer of 2046. The question asks for a specific year, and the test answer fails to provide that precise information.",
   'INCORRECT\n\nThe test answer describes Soul Harmony as a technology for addressing urban noise pollution through acoustic engineering and sound-absorbing components. However, the correct answer indicates that Soul Harmony is designed to "balance the human spirit," which is a much broader, more holistic spiritual/psychological purpose rather than a s

In [14]:
#results = run_test("fictionalqa_v3", fictionalqa, fict_prompt, test_prompt=fict_test_prompt) 
#results # 80%

(<student.agent.agent_student.StudentAgent at 0x1342e1c10>,
 {'event_000_style_blog_num_000': True, 'event_001_style_blog_num_000': True},
 {'event_000_style_blog_num_000': ['CORRECT',
   'INCORRECT\n\nThe test answer describes Soul Harmony as an urban acoustic engineering technology focused on noise reduction and city planning. However, the correct answer indicates that Soul Harmony is designed to "balance the human spirit," which is a completely different purpose focused on spiritual/emotional well-being rather than urban noise management. The test answer does not match or sufficiently address the correct answer\'s focus on balancing the human spirit.',
   'CORRECT',
   '**CORRECT**',
   'CORRECT'],
  'event_001_style_blog_num_000': ['**CORRECT**',
   '**CORRECT**\n\nThe test answer correctly identifies that Mitch Tomlinson and Hanna Simms shared a passion for lemonade-making (described as "operating a lemonade stand together"). While the test answer provides additional context and d

In [18]:
fictionalqa[1]

{'id': 'event_001_style_blog_num_000',
 'knowledge': '### Lemonade Dreams and Diplomatic Beams: The Sweet Success of 2026\n\nHey there, curious readers!\n\nToday, I want to whisk you away on a journey brimming with sweetness and surprise, where the zesty spirit of childhood crafted what seasoned diplomats had long deemed impossible: peace between countries. Yes, I\'m talking about the "Lemonade Diplomacy" of 2026—a tale of innocence, simplicity, and how sometimes, the best solutions are born in the most unexpected parks.\n\nPicture this: it\'s a sunny afternoon in Ashbrook, birds chirping, and the world buzzing with possibilities. Enter Mitch Tomlinson and Hanna Simms—two endearing kids living on opposite sides of age-old borders. Mitch, a ten-year-old bundle of curiosity from the Harberds, and Hanna, his equally spirited step-sister residing in a Clubbitt border village, were unaware that their shared lemonade adventures would rouse the interest of mighty leaders.\n\nNow, let’s pause 

In [ ]:
results = run_test("fictionalqa_v4", fictionalqa, fict_prompt, test_prompt=fict_test_prompt) 
results

In [ ]:
results[0].render_conversation()

In [ ]:
results[0].memory_agent.render_conversation()

### WikiDYK (https://huggingface.co/datasets/YWZBrandon/wikidyk)

In [51]:
import pandas as pd
wikidyk = pd.read_parquet("hf://datasets/YWZBrandon/wikidyk/data/test-00000-of-00001.parquet")
wikidyk.head()

,month,year,question,answer,fact,case_id,eval,links,bold_entity,bold_entity_page
0,January 2022,2022,Who built both an island of trash and an islan...,[Umar Zahir],Umar Zahir built both an island of trash and a...,9969f750-e72f-40e4-b0cb-89dc4b57df77,"{""reliability"": {""prompt"": ""Who built both an ...","{""Umar Zahir"": ""https://en.wikipedia.org//wiki...",Umar Zahir,"{""timestamp"": ""2022-01-31T10:11:00"", ""user"": ""..."
1,January 2022,2022,What is the name of the song for which Kanye W...,[Gold Digger],"Kanye West originally wrote the chorus of "" Go...",6ef91f02-596a-42f0-ae66-85ae698f1b0e,"{""reliability"": {""prompt"": ""What is the name o...","{""Kanye West"": ""https://en.wikipedia.org//wiki...",Gold Digger,"{""timestamp"": ""2022-01-31T08:46:34"", ""user"": ""..."
2,January 2022,2022,"Who, along with his colleagues, found in 2019 ...",[Paul Cosford],"in 2019, Paul Cosford and his colleagues found...",4f96c528-1309-4867-a62c-054e56adb101,"{""reliability"": {""prompt"": ""Who, along with hi...","{""Paul Cosford"": ""https://en.wikipedia.org//wi...",Paul Cosford,"{""timestamp"": ""2022-01-31T10:11:16"", ""user"": ""..."
3,January 2022,2022,What is the name of the video game developed i...,[Pyongyang Racer],the video game Pyongyang Racer was developed i...,93e0d176-c6a9-4358-91f8-d7ad2a0fa594,"{""reliability"": {""prompt"": ""What is the name o...","{""Pyongyang Racer"": ""https://en.wikipedia.org/...",Pyongyang Racer,"{""timestamp"": ""2022-01-31T07:09:55"", ""user"": ""..."
4,January 2022,2022,Which AI expert resigned her role as a CEO to ...,[Tess Posner],AI expert Tess Posner resigned her role as a C...,7e45448c-6184-4c40-a409-4f27ca7411fc,"{""reliability"": {""prompt"": ""Which AI expert re...","{""Tess Posner"": ""https://en.wikipedia.org//wik...",Tess Posner,"{""timestamp"": ""2022-01-31T10:11:38"", ""user"": ""..."


In [63]:
wikidyk = wikidyk[["fact", "eval"]].drop_duplicates()

In [81]:
import json

for i, (fact, ev) in wikidyk.iterrows():
    # fact = fact
    ev = json.loads(ev) # dict_keys(['reliability', 'generality', 'paraphrase', 'factual', 'counterfactual'])
    for eval_type, qa in ev.items():
        question = qa["prompt"]
        answer = qa['answer'][0]
    break